# Planet Finder: the daily search

One search day of the Planet Finder, run by RUNNER's scheduler from the public repository:

- **fast**, **deep** and **faint** queues on every core;
- live progress and a heartbeat to the site's monitor;
- candidates vetted, merged and sent to the API (`POST /finder/ingest`).

It finds **candidates**, never planets.

The search ledger (every star already searched) is saved as this notebook's **output** (`ph-state/`). This notebook reads its own previous output as an **input**, so no star is searched twice.

Setup: `kaggle/README.md` in the repository. Needs **Internet on** and two secrets: `PH_INGEST_TOKEN` and `PH_API_URL`.

In [ ]:
# Settings. PH_REPO_REF: the branch, tag or commit of the repository to run.
PH_REPO_URL = "https://github.com/anastasiadisfoivos-hub/planet-hunter.git"
PH_REPO_REF = "main"
PH_SESSION_HOURS = 11.5  # everything, search + wrap-up + save, ends by then (Kaggle stops a session at 12 h)

import os
import time

os.environ.setdefault("PH_SESSION_STARTED_AT", str(time.time()))
os.environ["PH_SESSION_HOURS"] = os.environ.get("PH_SESSION_HOURS_OVERRIDE") or str(PH_SESSION_HOURS)
PH_REPO_REF = os.environ.get("PH_REPO_REF_OVERRIDE") or PH_REPO_REF

In [ ]:
# Secrets: Add-ons > Secrets. They go into this kernel's environment only; nothing prints them.
try:
    from kaggle_secrets import UserSecretsClient
except ImportError:  # not on Kaggle (the local dry run): they come from the environment
    UserSecretsClient = None
if UserSecretsClient is not None:
    secrets = UserSecretsClient()
    for name in ("PH_INGEST_TOKEN", "PH_API_URL"):
        try:
            os.environ[name] = secrets.get_secret(name)
        except Exception as exc:
            print(f"secret {name}: not available ({type(exc).__name__}); attach it under Add-ons > Secrets")
for name in ("PH_INGEST_TOKEN", "PH_API_URL"):
    print(f"{name}: {'set' if os.environ.get(name) else 'MISSING'}")
print("API:", os.environ.get("PH_API_URL") or "none (nothing is posted or ingested)")

In [ ]:
# The repository at PH_REPO_REF (a shallow fetch; a commit hash works too).
import subprocess
from pathlib import Path

root = Path(os.environ.get("PH_KAGGLE_ROOT") or "/tmp/planet-hunter")
repo = root / "home" / "repo"
repo.mkdir(parents=True, exist_ok=True)
git = lambda *a: subprocess.run(["git", "-C", str(repo), *a], check=True, capture_output=True, text=True).stdout
if not (repo / ".git").exists():
    git("init", "-q")
    git("remote", "add", "origin", PH_REPO_URL)
git("fetch", "-q", "--depth=1", "origin", PH_REPO_REF)
git("checkout", "-q", "--force", "FETCH_HEAD")
print(f"{PH_REPO_REF} at", git("log", "-1", "--format=%h %cI %s").strip())
if not (repo / "kaggle" / "kaggle_run.py").exists():
    raise SystemExit(f"{PH_REPO_REF} has no kaggle/kaggle_run.py: set PH_REPO_REF to a ref that has kaggle/")

In [ ]:
# The search day: restore the ledger, install, search, vet, ingest, save the ledger. Runs for up to PH_SESSION_HOURS.
import sys

proc = subprocess.Popen([sys.executable, "-u", str(repo / "kaggle" / "kaggle_run.py")], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, "PH_REPO_DIR": str(repo)})
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:  # Stop: the scheduler stops cleanly and the ledger is still saved
    import signal
    proc.send_signal(signal.SIGINT)
code = proc.wait()
print("exit", code)

In [ ]:
# What this run saved (the next run's input).
import json

manifest = Path(os.environ.get("PH_KAGGLE_OUT") or "/kaggle/working") / "ph-state" / "manifest.json"
if not manifest.exists():
    raise SystemExit(f"nothing saved (exit {code}): the run stopped before the search; see the log above")
m = json.loads(manifest.read_text())
print(json.dumps({k: m.get(k) for k in ("saved_at", "run_id", "state", "done_total", "counts", "restored_from",
                                        "restored_done_total", "scheduler_exit", "error")}, indent=1))
if code != 0 or m.get("error"):
    raise SystemExit(f"the run did not finish cleanly (exit {code}); the ledger was saved all the same")